# 台指期

In [2]:
from datetime import datetime, timedelta
import os
import pandas as pd
import numpy as np

# setting #######################################################
path_TX = os.path.join(os.path.join(os.path.abspath(os.path.dirname(os.getcwd()))), 'history_data','tw','stock_price', 'tx.csv')     # 有過去資料的檔案
path_TXtech = os.path.join(os.path.join(os.path.abspath(os.path.dirname(os.getcwd()))), 'history_data','tw','stock_price', 'txtech.csv')     # 有過去資料的檔案

folder_raw = os.path.join(os.path.join(os.path.abspath(os.path.dirname(os.getcwd()))), 'history_data','tw','TX_RAW')     # 有過去資料的檔案
#################################################################


# 轉換 raw_data 成 每日資料

In [51]:
df_tx = pd.DataFrame()

# 只有日盤交易時間
for root, dirs, files in os.walk(folder_raw):
    # 排序檔案
    files.sort()
    for file in files:
        print(file)
        if file.endswith('.csv'):
            path_raw = os.path.join(root, file)
            # print(path_raw)    
            
            # 讀取資料 TX 
            raw_data = pd.read_csv(path_raw, encoding='big5')
            raw_data = raw_data.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
            raw_data = raw_data[raw_data['商品代號'] == 'TX']

            # 取得當月契約（最小的到期月份）
            # 轉換到期月份(週別) 欄位為 YYYY-MM 格式
            raw_data = raw_data.dropna(subset=['到期月份(週別)'])
            raw_data['到期月份(週別)'] = raw_data['到期月份(週別)'].astype(str)
            raw_data = raw_data[raw_data['到期月份(週別)'].str.match(r'^\d{6}$')]
            raw_data['到期月份(週別)'] = pd.to_datetime(raw_data['到期月份(週別)'], format='%Y%m', errors='coerce').dt.to_period('M')
            raw_data = raw_data.dropna(subset=['到期月份(週別)'])
            raw_data = raw_data[raw_data['到期月份(週別)'] == raw_data['到期月份(週別)'].min()]

            # 轉換成交日期和成交時間為完整 datetime
            raw_data = raw_data.dropna(subset=['成交日期', '成交時間'])
            raw_data['成交日期'] = raw_data['成交日期'].astype(str)
            raw_data['成交時間'] = raw_data['成交時間'].astype(str).str.zfill(6)  # 確保時間為 6 碼
            raw_data['成交日期時間'] = pd.to_datetime(raw_data['成交日期'] + raw_data['成交時間'], format='%Y%m%d%H%M%S', errors='coerce')

            # 取得 08:45:00 ~ 13:45:00 之間的資料
            start_time = datetime.strptime('09:00:00', '%H:%M:%S').time()
            end_time = datetime.strptime('13:30:00', '%H:%M:%S').time()
            raw_data = raw_data[(raw_data['成交日期時間'].dt.time >= start_time) & (raw_data['成交日期時間'].dt.time <= end_time)]
            raw_data = raw_data.drop(columns=['成交日期', '成交時間'])

            # 顯示結果
            # display(raw_data.head())
            # display(raw_data.tail())
            # print(len(raw_data))
            
            # 取得每日的開高低收量
            daily_data = raw_data.groupby(raw_data['成交日期時間'].dt.date).agg({
                '成交價格': ['first', 'max', 'min', 'last'],
                '成交數量(B+S)': 'sum'  # 正確的 Volume 計算方式
            })
            
            # 重新命名欄位
            daily_data.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
            daily_data.index.name = 'Date'
            daily_data.index = pd.to_datetime(daily_data.index)
            daily_data.index = daily_data.index.strftime('%Y%m%d')
            
            df_tx = pd.concat([df_tx, daily_data])
            # display(df_tx.head())
            # break
            
# 儲存 df_tx
df_tx.to_csv(path_TX, encoding='utf-8')

Daily_2023_05_10.csv
Daily_2023_05_11.csv
Daily_2023_05_12.csv
Daily_2023_05_15.csv
Daily_2023_05_16.csv
Daily_2023_05_17.csv
Daily_2023_05_18.csv
Daily_2023_05_19.csv
Daily_2023_05_22.csv
Daily_2023_05_23.csv
Daily_2023_05_24.csv
Daily_2023_05_25.csv
Daily_2023_05_26.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_79026/3088142524.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(path_raw, encoding='big5')


Daily_2023_05_29.csv
Daily_2023_05_30.csv
Daily_2023_05_31.csv
Daily_2023_06_01.csv
Daily_2023_06_02.csv
Daily_2023_06_05.csv
Daily_2023_06_06.csv
Daily_2023_06_07.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_79026/3088142524.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(path_raw, encoding='big5')


Daily_2023_06_08.csv
Daily_2023_06_09.csv
Daily_2023_06_12.csv
Daily_2023_06_13.csv
Daily_2023_06_14.csv
Daily_2023_06_15.csv
Daily_2023_06_16.csv
Daily_2023_06_19.csv
Daily_2023_06_20.csv
Daily_2023_06_21.csv
Daily_2023_06_26.csv
Daily_2023_06_27.csv


...

/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_79026/3088142524.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(path_raw, encoding='big5')


Daily_2024_12_23.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_79026/3088142524.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(path_raw, encoding='big5')


Daily_2024_12_24.csv
Daily_2024_12_25.csv
Daily_2024_12_26.csv
Daily_2024_12_27.csv
Daily_2024_12_30.csv
Daily_2024_12_31.csv


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_79026/3088142524.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_data = pd.read_csv(path_raw, encoding='big5')


# 算技術指標

In [3]:

df = pd.read_csv(path_TX, encoding='utf-8')

# 計算移動均線 (SMA)
df['MA5'] = df['Close'].rolling(window=5).mean()
df['MA10'] = df['Close'].rolling(window=10).mean()
df['MA20'] = df['Close'].rolling(window=20).mean()

# 計算指數移動平均線 (EMA)
df['EMA5'] = df['Close'].ewm(span=5, adjust=False).mean()
df['EMA12'] = df['Close'].ewm(span=12, adjust=False).mean()
df['EMA26'] = df['Close'].ewm(span=26, adjust=False).mean()
df['EMA60'] = df['Close'].ewm(span=26, adjust=False).mean()


# 計算MACD和DIF
df['DIF'] = df['EMA12'] - df['EMA26']
df['MACD'] = df['DIF'].ewm(span=9, adjust=False).mean()


# 計算RSI
def calculate_rsi(series, period=14):
    delta = series.diff(1)
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df['RSI'] = calculate_rsi(df['Close'])

# 計算KDJ
low_list = df['Low'].rolling(9, min_periods=9).min()
low_list.fillna(value=df['Low'].expanding().min(), inplace=True)
high_list = df['High'].rolling(9, min_periods=9).max()
high_list.fillna(value=df['High'].expanding().max(), inplace=True)
rsv = (df['Close'] - low_list) / (high_list - low_list) * 100


# 波動率
def historical_volatility(prices, window=10, trading_days=252):
    """
    計算歷史波動率 (Historical Volatility)

    :param prices: 一個包含價格數據的列表或Numpy數組
    :param window: 計算波動率的視窗期 (默認是252天)
    :param trading_days: 年化的交易天數，默認是252
    :return: 年化波動率
    """
    prices = np.asarray(prices)
    # 計算對數收益率
    log_returns = np.log(prices[1:] / prices[:-1])

    # 計算對數收益率的標準差
    std_dev = np.std(log_returns[-window:])

    # 年化波動率
    annualized_volatility = std_dev * np.sqrt(trading_days)
    
    return annualized_volatility

df['Volatility'] = df['Close'].rolling(window=252).apply(historical_volatility, raw=True)

# 初始值設定
df['K'] = 50
df['D'] = 50
for i in range(1, len(df)):
    df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
    df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']

# # Rename columns
# df.rename(columns={'日期': 'Date', '開盤指數': 'Open', '最高指數': 'High', '最低指數': 'Low', '收盤指數': 'Close'}, inplace=True)


display(df.head())

# 顯示DataFrame
df.to_csv(path_TXtech, encoding='utf-8', index=False)


/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_82952/615847271.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '38.69239013933547' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'K'] = (2/3) * df.loc[df.index[i-1], 'K'] + (1/3) * rsv[i]
/var/folders/xc/rjj8m66j6xs5pgcp1hc7g2kr0000gn/T/ipykernel_82952/615847271.py:68: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46.230796713111815' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df.index[i], 'D'] = (2/3) * df.loc[df.index[i-1], 'D'] + (1/3) * df.loc[df.index[i], 'K']


,Date,Open,High,Low,Close,Volume,MA5,MA10,MA20,EMA5,EMA12,EMA26,EMA60,DIF,MACD,RSI,Volatility,K,D
0,19980721,8131,8131,8036,8045,208,NaN,NaN,NaN,8045.000000,8045.000000,8045.000000,8045.000000,0.000000,0.000000,NaN,NaN,50.000000,50.000000
1,19980722,7950,7950,7820,7870,360,NaN,NaN,NaN,7986.666667,8018.076923,8032.037037,8032.037037,-13.960114,-2.792023,NaN,NaN,38.692390,46.230797
2,19980723,7830,7859,7802,7825,190,NaN,NaN,NaN,7932.777778,7988.372781,8016.700960,8016.700960,-28.328179,-7.899254,NaN,NaN,28.125221,40.195605
3,19980724,7750,7835,7740,7835,340,NaN,NaN,NaN,7900.185185,7964.776969,8003.241630,8003.241630,-38.464661,-14.012335,NaN,NaN,26.849039,35.746749
4,19980727,7830,7835,7780,7786,150,7872.2,NaN,NaN,7862.123457,7937.272820,7987.149657,7987.149657,-49.876838,-21.185236,NaN,NaN,21.820928,31.104809
